# Problemstellung: Optimale Einsatzplanung für Kundendienst

Sie betreiben einen mobilen Kundendienst mit mehreren Mitarbeitenden, die jeweils mit
einem Servicefahrzeug unterwegs sind. Die Mitarbeitenden starten an vorgegebenen
Depots. Zwischen allen Standorten sind die Fahrzeiten bekannt (siehe Daten). Ihr
Unternehmen muss innerhalb eines Planungshorizonts von 600 Minuten alle Aufträge
ausführen.

## Datenmodell

Die bereitgestellte Excel-Datei enthält folgende Tabellen:
- TECHS: Techniker/innen mit Startdepot, Stundensatz und Qualifikationen.
- JOBS: Aufträge mit Standort, geplanter Startzeit, Dauer, Anforderungsprofil, VIP-
Flag und Strafzahlung pro Minute Verspätung.
- TIME: Fahrzeitmatrix in Minuten zwischen allen Knoten (Depots und
Kundenstandorte).
- PARAMETERS: Globale Parameter (Schichtende, Rüstzeit, etc.).

## Regeln und Annahmen
- Alle Aufträge müssen durchgeführt werden; unrentable Aufträge dürfen nicht abgelehnt werden.
- Ein Auftrag darf nicht vor seiner geplanten Startzeit beginnen (Früherer Beginn istverboten). Warten ist erlaubt.
verboten). Warten ist erlaubt.
- Beginnt ein Auftrag verspätet, fällt eine Kompensationszahlung an: penalty_eur_per_min × Verspätungsminuten (pro Auftrag).
- Die Bearbeitung eines Auftrags dauert duration_min Minuten (aus der JOBS-Tabelle).
- Zwischen zwei Aufträgen ist eine Rüst-/Dokumentationszeit von 10 Minuten einzuplanen.
- Qualifikationen (Skills): DELIVERY kann jede Person übernehmen; ELECTRIC nur Senior_Elektriker oder Meister; PLUMB nur Senior_Klempner oder Meister; Meister
Senior_Elektriker oder Meister; PLUMB nur Senior_Klempner oder Meister; Meister
kann alles.
- Schichtende: Nach 600 Minuten ist Schluss (nicht individuell pro Mitarbeitersondern nach Arbeitsbeginn). Danach sollen keine Aufträge mehr bearbeitet sondern nach Arbeitsbeginn). Danach sollen keine Aufträge mehr bearbeitet werden.
- Betriebskosten: Kosten pro Stunde gemäß TECHS (Vollkostenmodell für Fahr-, Warte- und Arbeitszeit). Die Techniker werden bezahlt für die Zeit zwischen persönlichem Schichtbeginn und -ende.
- Auftrags-Erlös: pro Auftrag revenue_eur (Deckungsbeitrag) gemäß JOBS.
- Wir gehen davon aus, dass die Fahrzeiten akkurat sind und es keinen Stau oder
sonstige Verzögerungen gibt.

# Initialisierung

Es werden zunächst benötigte Module geladen sowie die Daten aus der Excel-Datei eingelesen und in geeigneten Datenstrukturen gespeichert. Dies umfasst die Informationen über die Techniker, Aufträge, Fahrzeiten und Parameter.

In [1]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import time
from deap import base, creator, tools, algorithms

In [2]:
# Laden der Daten
data = pd.read_excel('field_service_ea_600min_v2.xlsx', sheet_name=None)
# Speicherung der Daten in geeigneten Strukturen
technicians = data['TECHS']
technicians['skills'] = technicians['skills'].apply(lambda x: set(map(str, x.split(','))))
jobs = data['JOBS']
time = data['TIME']
nodes = data['NODES']
parameters = data['PARAMETERS'].set_index('name')['value'].to_dict()

penalty = 100000

# 1. Entwerfen Sie einen evolutionären Algorithmus
## 1a & 1b: Repräsentation, Strategie und Begründung

### 1. Repräsentation des Genoms
**Wahl:** Das Genom wird als Liste von Tupeln repräsentiert: `[(Job_ID_1, Tech_ID_x), (Job_ID_2, Tech_ID_y), ...]`.

**Begründung:** Diese direkte Kodierung ordnet jeden Auftrag genau einem Techniker zu. Da alle Aufträge ausgeführt werden müssen (und wir keine auslassen dürfen), bleibt die Länge des Genoms konstant. Durch die Vorfilterung der Skills (nur qualifizierte Techniker werden bei der Initialisierung und Mutation zugewiesen) reduzieren wir den Suchraum enorm und verhindern, dass der Algorithmus Zeit mit von vornherein ungültigen Lösungen verschwendet.

### 2. Vererbungs-/Mutationsstrategie
**Wahl Crossover (`cxTwoPoint`):** Wir verwenden einen klassischen Zwei-Punkt-Crossover. 

**Begründung:** Ein Zwei-Punkt-Crossover tauscht zusammenhängende Blöcke von Job-Zuweisungen zwischen zwei Eltern. Da die Initialisierung (Heuristik) die Jobs zeitlich sortiert, bedeutet ein Crossover, dass zeitlich zusammenhängende Einsatzpläne eines guten Elternteils als Block vererbt werden können.

**Wahl Mutation:** Die Mutation wählt zufällig einen Job aus und weist ihn einem *anderen, aber qualifizierten* Techniker zu. 

**Begründung:** Dies erlaubt es dem Algorithmus, lokale Optima zu verlassen, indem einzelne Jobs umverteilt werden, um beispielsweise Überstunden bei einem überlasteten Techniker abzubauen.

### 3. Evaluationsfunktion (Fitness)
**Wahl:** Die Fitnessfunktion berechnet den Reingewinn (`Deckungsbeitrag - Strafen - Arbeitskosten`).

**Begründung:** Das Ziel des Unternehmens ist die Gewinnmaximierung. Harte Restriktionen (fehlende Qualifikation, Überschreiten der 600 Minuten) werden mit enormen Strafzahlungen (`penalty = 100000`) sanktioniert, sodass solche Individuen schnell aussortiert werden.

## Aufgabe 2b & 2c: Beschreibung der Methoden & Spezifisch vs. Generisch

**Aufgaben der verschiedenen Methoden:**
*   `create_individual`: Erzeugt initiale Einsatzpläne. Für einen schnellen Start verwenden die ersten 25 Individuen eine Greedy-Heuristik (nächster verfügbarer Techniker), der Rest wird zufällig generiert, um die Diversität der Population sicherzustellen.
*   `evaluate`: Simuliert den Arbeitstag für jedes Individuum. Berechnet sequenziell Fahrzeiten, Wartezeiten, Arbeitszeiten und Rüstzeiten pro Techniker und aggregiert die daraus resultierenden Kosten und Erlöse zu einem Gesamtgewinn.
*   `mutate_reassign_technician`: Sorgt für Diversität im Verlauf der Evolution, indem Techniker bei einzelnen Jobs gewechselt werden (unter Wahrung der Skill-Anforderungen).

**Spezifisch für diese Aufgabenstellung:**
Die Repräsentation der Individuen (Job-Techniker-Tupel), die `evaluate`-Funktion (Simulierung der Schicht, Rüstzeiten, VIP-Penalties) und die Initialisierung (Skill-Mapping, Heuristik) sind hochspezifisch für das Field-Service-Problem.

**Eher generisch:**
Die Architektur des EAs an sich (DEAP-Framework), die Definition von Population, Selektion (`selTournament`) und der Crossover-Operator (`cxTwoPoint`) sind generische Standard-Werkzeuge, die auf viele Optimierungsprobleme anwendbar sind.

In [3]:
# Vorberechnung der qualifizierten Techniker pro Auftrag
valid_technicians_for_job = {}
for _, job in jobs.iterrows():
    req_skill = job['required_skill']
    valid_techs = [t.id for t in technicians.itertuples() if req_skill in t.skills]
    valid_technicians_for_job[job['id']] = valid_techs

create_counter = 0

# Initialisierung eines Individuums (Heuristic Seeding & Skill-Aware)
def create_individual():
    global create_counter
    create_counter += 1
    
    # Die ersten 25 Individuen basierend auf einer Nearest-Neighbor / greedy Zeit-Heuristik erstellen
    if create_counter <= 25:
        individual = []
        tech_status = {t.id: {'time': 0, 'loc': t.start_node} for t in technicians.itertuples()}
        
        sorted_jobs = jobs.sort_values('planned_start_min')
        assignments = {}
        
        for _, job in sorted_jobs.iterrows():
            job_id = job['id']
            valid_techs = valid_technicians_for_job[job_id]
            
            # 20% Randomness, um Varianz zwischen den "smarten" Start-Individuen zu haben
            if random.random() < 0.2:
                best_tech = random.choice(valid_techs)
            else:
                best_tech = valid_techs[0]
                min_cost = float('inf')
                
                for tech_id in valid_techs:
                    loc = tech_status[tech_id]['loc']
                    # Fahrzeit aus der Zeitmatrix auslesen
                    travel_time = time.loc[time["From/To"] == loc, job['node_id']].values[0]
                    arrival = tech_status[tech_id]['time'] + travel_time
                    
                    # Kosten = Fahrzeit + Wartezeit
                    wait = max(0, job['planned_start_min'] - arrival)
                    cost = travel_time + wait
                    
                    # Extrem hart bestrafen, falls der Techniker durch vorherige Jobs verspätet ankommen würde
                    if arrival > job['planned_start_min']:
                        cost += (arrival - job['planned_start_min']) * 100
                        
                    if cost < min_cost:
                        min_cost = cost
                        best_tech = tech_id
                        
            assignments[job_id] = best_tech
            
            # Status des gewählten Technikers updaten, um ihn für den nächsten Job zu tracken
            travel_time = time.loc[time["From/To"] == tech_status[best_tech]['loc'], job['node_id']].values[0]
            new_time = max(tech_status[best_tech]['time'] + travel_time, job['planned_start_min'])
            tech_status[best_tech]['time'] = new_time + job['duration_min'] + parameters['SETUP_BETWEEN_JOBS_MIN']
            tech_status[best_tech]['loc'] = job['node_id']
            
        # Den Vektor passend zur ursprünglichen JOBS-Tabelle formatieren (für korrekten Crossover)
        for job_id in jobs['id']:
            individual.append((job_id, assignments[job_id]))
            
        return individual

    else:
        # Komplett zufällige (aber skill-validierte) Initialisierung für den Rest der Population
        individual = []
        for job_id in jobs['id']:
            technician_id = random.choice(valid_technicians_for_job[job_id])
            individual.append((job_id, technician_id))
            
        return individual

In [4]:
# Vorberechnung für schnelle Lookups in der Fitness-Funktion (O(1) statt O(N) Pandas-Suchen)
travel_times_dict = {}
for _, row in time.iterrows():
    from_loc = row['From/To']
    for to_loc in time.columns[1:]:
        travel_times_dict[(from_loc, to_loc)] = row[to_loc]

job_dict = jobs.set_index('id').to_dict('index')
tech_dict = {t.id: t for t in technicians.itertuples()}

# Fitness-Funktion zur Gewinnmaximierung
def evaluate(individual):
    # Berechnung des Gesamtgewinns basierend auf der Vergütung und den Kosten der Techniker
    total_revenue = 0
    
    # Schnelles Gruppieren der Aufträge pro Techniker
    assigned_to_tech = {t_id: [] for t_id in tech_dict.keys()}
    for job_id, tech_id in individual:
        assigned_to_tech[tech_id].append(job_id)
        
    for tech_id, assigned_jobs in assigned_to_tech.items():
        revenue_per_technician = 0
        current_time = 0
        paid_working_time = 0
        
        technician = tech_dict[tech_id]
        current_location = technician.start_node
        
        if not assigned_jobs:
            continue

        # Ultraschnelles Sortieren der Jobs nach geplanter Startzeit mithilfe des Dictionaries
        assigned_jobs.sort(key=lambda j: job_dict[j]['planned_start_min'])
                
        for i, job_id in enumerate(assigned_jobs):
            job_info = job_dict[job_id]
            
            # Berechnung der Fahrzeit zum nächsten Auftrag
            job_location = job_info['node_id']
            travel_time = travel_times_dict.get((current_location, job_location), 0)
            current_time += travel_time
            paid_working_time += travel_time
            
            # Wartezeit vor Ort oder Abfahrt verzögern (beim ersten Job)
            planned_start_min = job_info['planned_start_min']
            
            if i == 0 and current_time < planned_start_min:
                # Techniker wartet zu Hause, fährt erst passend los
                current_time = planned_start_min
            else:
                if current_time < planned_start_min:
                    # Wartezeit vor Ort (gilt als bezahlte Arbeitszeit)
                    wait_time = planned_start_min - current_time
                    current_time = planned_start_min
                    paid_working_time += wait_time
                elif current_time > planned_start_min:
                    compensation = (current_time - planned_start_min) * job_info['penalty_eur_per_min']
                    revenue_per_technician -= compensation  # Abzug für verspäteten Start

            # Berechnung der Arbeitszeit für den Auftrag
            current_time += job_info['duration_min']
            paid_working_time += job_info['duration_min']
            current_location = job_location
            
            # Skills prüfen
            required_skill = job_info['required_skill']
            if required_skill in technician.skills:
                revenue_per_technician += job_info['revenue_eur']
            else:
                revenue_per_technician -= penalty  # Strafe für unqualifizierte Techniker
            
            # Rüst-/Dokumentationszeit zwischen den Aufträgen
            if i < len(assigned_jobs) - 1:
                current_time += parameters['SETUP_BETWEEN_JOBS_MIN']
                paid_working_time += parameters['SETUP_BETWEEN_JOBS_MIN']
                
        # Berücksichtigung der Arbeitszeit des Technikers
        if current_time > parameters['SHIFT_END_MIN']:
            revenue_per_technician -= penalty  # Strafe für Überschreitung der Schicht
            
        # Abzug der Kosten basierend auf der tatsächlichen Arbeitszeit
        revenue_per_technician -= (technician.cost_eur_per_h / 60) * paid_working_time  
            
        total_revenue += revenue_per_technician    
        
    return total_revenue,

In [5]:
# Selektions-, Crossover- und Mutationsoperatoren

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)
creator.create("Population", list)
toolbox = base.Toolbox()
toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, creator.Population, toolbox.individual)
toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)

# Mutation: Weist einem zufälligen Auftrag einen anderen *qualifizierten* Techniker zu
def mutate_reassign_technician(individual, indpb=0.1):
    for i, (job_id, technician_id) in enumerate(individual):
        if random.random() < indpb:
            valid_techs = valid_technicians_for_job[job_id]
            other_technicians = [t for t in valid_techs if t != technician_id]
            if other_technicians:
                individual[i] = (job_id, random.choice(other_technicians))
    return (individual,)

toolbox.register("mutate", mutate_reassign_technician, indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=3)

In [6]:
# Generationaler Ablauf des genetischen Algorithmus
def main():
    global create_counter
    create_counter = 0  # Counter immer für jeden Lauf auf null setzen
    
    random.seed(42)
    POP_SIZE = 1000
    NGEN = 1000
    
    population = toolbox.population(n=POP_SIZE)
    hof = tools.HallOfFame(5)
    
    # Statistiken für die Auswertung sammeln
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    stats.register("median", np.median)
    
    logbook = tools.Logbook()
    logbook.header = "gen", "evals", "avg", "min", "max", "std", "median"
    
    # Initiale Evaluation
    fits = toolbox.map(toolbox.evaluate, population)
    for fit, ind in zip(fits, population):
        ind.fitness.values = fit
        
    hof.update(population)
    record = stats.compile(population)
    logbook.record(gen=0, evals=len(population), **record)
    print(logbook.stream)
    
    # Echter Elitismus (Custom Loop): Übernimmt die besten n Individuen unverändert
    ELITISM_SIZE = 5 
    
    for gen in range(1, NGEN + 1):
        # Selektion der Eltern (Tournament)
        mating_pool = toolbox.select(population, k=POP_SIZE)
        mating_pool = list(map(toolbox.clone, mating_pool))
        
        # Crossover & Mutation (varAnd generiert Nachkommen)
        offspring = algorithms.varAnd(mating_pool, toolbox, cxpb=0.7, mutpb=0.3)
        
        # Fitness der veränderten Nachkommen neu berechnen
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fits = toolbox.map(toolbox.evaluate, invalid_ind)
        for fit, ind in zip(fits, invalid_ind):
            ind.fitness.values = fit
            
        hof.update(offspring)
        
        # ELITISMUS: Die besten ELITISM_SIZE Individuen aus der alten Generation pushen
        elites = tools.selBest(population, ELITISM_SIZE)
        
        # Survivor Selection: Fülle den Rest mit den besten der neuen Nachkommen auf
        population = elites + tools.selBest(offspring, POP_SIZE - ELITISM_SIZE)
        
        record = stats.compile(population)
        logbook.record(gen=gen, evals=len(invalid_ind), **record)
        if gen % 50 == 0:
            print(logbook.stream)
            
    best_individual = hof[0]
    print("Best individual overall:", best_individual)
    print("Best fitness overall:", best_individual.fitness.values[0])
    return best_individual, logbook
    
best_individual, logbook = main()

gen	evals	avg    	min    	max    	std    	median 
0  	1000 	-165371	-421811	2641.58	82314.8	-203034
1  	806  	-124133 	-314370	2641.58	68893.5	-106474 
2  	813  	-100577 	-310652	2641.58	69490.2	-104240 
3  	810  	-87648  	-308135	2641.58	70536.3	-102913 
4  	809  	-77484  	-309510	2676.43	70129.4	-101729 
5  	763  	-63314.2	-308058	2682.38	69278.5	-98235.6
6  	784  	-52764.6	-304916	2682.38	66181.7	-2932.99
7  	781  	-48652  	-212434	2682.38	64131.6	-2165.92
8  	796  	-44086.4	-217261	2682.38	63538.9	-1058.8 
9  	799  	-33716.5	-209704	2682.38	58369.5	88.9583 
10 	770  	-18510.7	-206499	2682.38	43824.7	1116.77 
11 	779  	-11999.1	-201704	2682.38	36248.8	1784.85 
12 	786  	-8644.19	-201976	2701.28	33135.2	2224.98 
13 	795  	-7081.16	-199978	2729.98	29973.8	2479.58 
14 	789  	-6113.26	-104286	2729.98	28050.5	2641.58 
15 	803  	-7878.45	-199366	2765.25	30928.2	2641.58 
16 	799  	-8056.11	-200190	2765.25	31184.2	2676.43 
17 	768  	-7706.46	-200948	2779.18	32348.4	2682.38 
18 	783  	-8262.

# Aufgabe 3: Visualisieren, diskutieren und interpretieren Sie die Ergebnisse

In [7]:
import plotly.express as px
import plotly.graph_objects as go
import datetime

def visualize_time_plan(individual):
    events = []
    # Dummy base date for plotting times
    base_time = datetime.datetime(2026, 1, 1, 8, 0)
    
    for y_idx, technician in enumerate(technicians.itertuples()):
        current_location = technician.start_node
        working_time = 0
        
        assigned_jobs_unsorted = [job_id for job_id, tech_id in individual if tech_id == technician.id]
        if not assigned_jobs_unsorted:
            continue
            
        assigned_jobs_df = jobs[jobs['id'].isin(assigned_jobs_unsorted)].sort_values(by='planned_start_min')
        assigned_jobs = assigned_jobs_df['id'].tolist()
        
        for i, job_id in enumerate(assigned_jobs):
            job = jobs[jobs['id'] == job_id].iloc[0]
            job_location = job['node_id']
            
            travel_row = time.loc[time["From/To"] == current_location, job_location]
            travel_time = travel_row.values[0] if not travel_row.empty else 0
            planned_start_min = job['planned_start_min']
            
            if i == 0 and (working_time + travel_time) < planned_start_min:
                working_time = planned_start_min - travel_time
            
            if travel_time > 0:
                events.append(dict(
                    Technician=technician.id,
                    Phase="Travel Time",
                    Start=base_time + datetime.timedelta(minutes=int(working_time)),
                    Finish=base_time + datetime.timedelta(minutes=int(working_time + travel_time)),
                    Info=f"From {current_location} to {job_location} ({travel_time}m)"
                ))
                working_time += travel_time
                
            if working_time < planned_start_min:
                wait_time = planned_start_min - working_time
                events.append(dict(
                    Technician=technician.id,
                    Phase="Wait Time",
                    Start=base_time + datetime.timedelta(minutes=int(working_time)),
                    Finish=base_time + datetime.timedelta(minutes=int(working_time + wait_time)),
                    Info=f"Waiting for Job {job_id} planned start ({wait_time}m)"
                ))
                working_time = planned_start_min
            elif working_time > planned_start_min:
                late_time = working_time - planned_start_min
                events.append(dict(
                    Technician=technician.id,
                    Phase="Late Gap (Penalty)",
                    Start=base_time + datetime.timedelta(minutes=int(planned_start_min)),
                    Finish=base_time + datetime.timedelta(minutes=int(working_time)),
                    Info=f"Late for Job {job_id} by {late_time}m"
                ))
                
            job_duration = job['duration_min']
            events.append(dict(
                Technician=technician.id,
                Phase="Job Execution",
                Start=base_time + datetime.timedelta(minutes=int(working_time)),
                Finish=base_time + datetime.timedelta(minutes=int(working_time + job_duration)),
                Info=f"Job {job_id} | Revenue: {job['revenue_eur']}€ | Skill: {job['required_skill']}"
            ))
            working_time += job_duration
            current_location = job_location
            
            if i < len(assigned_jobs) - 1:
                setup_time = parameters['SETUP_BETWEEN_JOBS_MIN']
                events.append(dict(
                    Technician=technician.id,
                    Phase="Setup Time",
                    Start=base_time + datetime.timedelta(minutes=int(working_time)),
                    Finish=base_time + datetime.timedelta(minutes=int(working_time + setup_time)),
                    Info=f"Setup ({setup_time}m)"
                ))
                working_time += setup_time

    if not events:
        print("No events to plot.")
        return

    df_events = pd.DataFrame(events)
    
    color_discrete_map = {
        "Job Execution": "#3498DB",
        "Travel Time": "#F39C12",
        "Setup Time": "#F1C40F",
        "Wait Time": "#BDC3C7",
        "Late Gap (Penalty)": "#E74C3C"
    }

    fig = px.timeline(df_events, x_start="Start", x_end="Finish", y="Technician", color="Phase",
                      color_discrete_map=color_discrete_map, hover_name="Info",
                      title="Modern Interactive Gantt Chart: Technician Schedule")
    
    # Adjust layout
    fig.update_yaxes(autorange="reversed")
    
    # Add vertical line for Shift End Max
    shift_end_dt = base_time + datetime.timedelta(minutes=int(parameters['SHIFT_END_MIN']))
    
    # FIX: Convert the datetime object to an epoch timestamp in milliseconds
    shift_end_ms = shift_end_dt.timestamp() * 1000 
    
    fig.add_vline(x=shift_end_ms, line_width=3, line_dash="dash", line_color="red", annotation_text="Shift End (600m)")
    
    # Format x-axis to show minutes elapsed intuitively (hide actual date)
    fig.update_xaxes(tickformat="%H:%M", title="Time of Day (Assumed 08:00 Start)")
    fig.update_layout(template="plotly_white", height=600)
    fig.show()


In [8]:
import plotly.graph_objects as go

def plot_fitness_evolution(logbook, title="Evolution of Population Fitness (Revenue across Generations)", generations_to_show=50):
    gen = logbook.select("gen")[:generations_to_show]
    fit_mins = logbook.select("min")[:generations_to_show]
    fit_avgs = logbook.select("avg")[:generations_to_show]
    fit_maxs = logbook.select("max")[:generations_to_show]
    fit_medians = logbook.select("median")[:generations_to_show]
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(x=gen, y=fit_maxs, mode='lines', name='Maximum Fitness', 
                             line=dict(color='#27AE60', width=2, dash='dot')))
    fig.add_trace(go.Scatter(x=gen, y=fit_avgs, mode='lines', name='Average Fitness', 
                             line=dict(color='#2980B9', width=3)))
    fig.add_trace(go.Scatter(x=gen, y=fit_mins, mode='lines', name='Minimum Fitness', 
                             line=dict(color='#E74C3C', width=2, dash='dash')))
    fig.add_trace(go.Scatter(x=gen, y=fit_medians, mode='lines', name='Median Fitness', 
                             line=dict(color='#8E44AD', width=2, dash='dot')))
    
    # Modernizing the layout layout
    fig.update_layout(
        title=title,
        xaxis_title="Generation",
        yaxis_title="Fitness (Expected Profit in €)",
        template="plotly_white",
        hovermode="x unified",
        legend=dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99)
    )
    
    # Zoom and pan enable naturally
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
    
    fig.show()


In [9]:
import plotly.graph_objects as go

def evaluate_detailed(individual):
    # Berechnung des Gesamtgewinns basierend auf der Vergütung und den Kosten der Techniker
    total_revenues = 0
    total_penalty = 0
    total_travel_cost = 0
    total_wait_cost = 0
    total_work_cost = 0
    total_setup_cost = 0
    
    assigned_to_tech = {t_id: [] for t_id in tech_dict.keys()}
    for job_id, tech_id in individual:
        assigned_to_tech[tech_id].append(job_id)
        
    for tech_id, assigned_jobs in assigned_to_tech.items():
        current_time = 0
        
        technician = tech_dict[tech_id]
        current_location = technician.start_node
        cost_per_min = technician.cost_eur_per_h / 60.0
        
        if not assigned_jobs:
            continue

        assigned_jobs.sort(key=lambda j: job_dict[j]['planned_start_min'])
                
        for i, job_id in enumerate(assigned_jobs):
            job_info = job_dict[job_id]
            job_location = job_info['node_id']
            
            travel_time = travel_times_dict.get((current_location, job_location), 0)
            current_time += travel_time
            total_travel_cost += travel_time * cost_per_min
            
            planned_start_min = job_info['planned_start_min']
            
            if i == 0 and current_time < planned_start_min:
                current_time = planned_start_min
            else:
                if current_time < planned_start_min:
                    wait_time = planned_start_min - current_time
                    current_time = planned_start_min
                    total_wait_cost += wait_time * cost_per_min
                elif current_time > planned_start_min:
                    compensation = (current_time - planned_start_min) * job_info['penalty_eur_per_min']
                    total_penalty += compensation

            current_time += job_info['duration_min']
            total_work_cost += job_info['duration_min'] * cost_per_min
            current_location = job_location
            
            required_skill = job_info['required_skill']
            if required_skill in technician.skills:
                total_revenues += job_info['revenue_eur']
            else:
                total_penalty += penalty
            
            if i < len(assigned_jobs) - 1:
                current_time += parameters['SETUP_BETWEEN_JOBS_MIN']
                total_setup_cost += parameters['SETUP_BETWEEN_JOBS_MIN'] * cost_per_min
                
        if current_time > parameters['SHIFT_END_MIN']:
            total_penalty += penalty
            
    summary = {
        'Auftrags-Einnahmen': total_revenues,
        'Arbeitszeit (Kosten)': -total_work_cost,
        'Reisezeit (Kosten)': -total_travel_cost,
        'Wartezeit (Kosten)': -total_wait_cost,
        'Rüstzeit (Kosten)': -total_setup_cost,
        'Strafen': -total_penalty
    }
    
    return summary

def plot_cost_breakdown(individual):
    summary = evaluate_detailed(individual)
    
    measure = ["relative"] * len(summary) + ["total"]
    x = list(summary.keys()) + ["Netto-Gewinn"]
    y = list(summary.values()) + [sum(summary.values())]
    text = [f"{v:,.0f} €" for v in y]
    
    fig = go.Figure(go.Waterfall(
        name="Profit Breakdown", orientation="v",
        measure=measure,
        x=x, textposition="outside", text=text,
        y=y,
        connector={"line": {"color": "rgb(63, 63, 63)", "width": 1, "dash": "dot"}},
        decreasing={"marker": {"color": "#E74C3C"}},
        increasing={"marker": {"color": "#2ECC71"}},
        totals={"marker": {"color": "#34495E"}}
    ))
    
    fig.update_layout(
        title=f"Financial Waterfall Breakdown (Net Profit: {y[-1]:,.2f} €)",
        showlegend=False,
        template="plotly_white",
        yaxis_title="Euro (€)"
    )
    fig.show()


In [10]:
import plotly.express as px

def plot_technician_utilization(individual):
    """
    Analyzes and visualizes the percentage breakdown of each technician's day 
    (Working vs Traveling vs Waiting vs Setup vs Idle/End of Shift).
    Helps us understand if routing is optimal and technicians are utilized.
    """
    stats = []
    
    assigned_to_tech = {t_id: [] for t_id in tech_dict.keys()}
    for job_id, tech_id in individual:
        assigned_to_tech[tech_id].append(job_id)
        
    for tech_id, assigned_jobs in assigned_to_tech.items():
        technician = tech_dict[tech_id]
        current_location = technician.start_node
        current_time = 0
        
        comp_travel = 0
        comp_wait = 0
        comp_work = 0
        comp_setup = 0

        if not assigned_jobs:
            stats.append({'Technician': technician.id, 'Task': 'Idle Day', 'Minutes': parameters['SHIFT_END_MIN']})
            continue

        assigned_jobs.sort(key=lambda j: job_dict[j]['planned_start_min'])
                
        for i, job_id in enumerate(assigned_jobs):
            job_info = job_dict[job_id]
            job_location = job_info['node_id']
            
            travel_time = travel_times_dict.get((current_location, job_location), 0)
            current_time += travel_time
            comp_travel += travel_time
            
            planned_start_min = job_info['planned_start_min']
            
            if i == 0 and current_time < planned_start_min:
                current_time = planned_start_min
            else:
                if current_time < planned_start_min:
                    wait_time = planned_start_min - current_time
                    current_time = planned_start_min
                    comp_wait += wait_time

            job_dur = job_info['duration_min']
            current_time += job_dur
            comp_work += job_dur
            current_location = job_location
            
            if i < len(assigned_jobs) - 1:
                setup = parameters['SETUP_BETWEEN_JOBS_MIN']
                current_time += setup
                comp_setup += setup
                
        comp_remaining = max(0, parameters['SHIFT_END_MIN'] - current_time)
        
        stats.append({'Technician': technician.id, 'Activity': 'Working', 'Minutes': comp_work})
        stats.append({'Technician': technician.id, 'Activity': 'Driving', 'Minutes': comp_travel})
        stats.append({'Technician': technician.id, 'Activity': 'Waiting', 'Minutes': comp_wait})
        stats.append({'Technician': technician.id, 'Activity': 'Setup', 'Minutes': comp_setup})
        if comp_remaining > 0:
            stats.append({'Technician': technician.id, 'Activity': 'Idle (Shift End)', 'Minutes': comp_remaining})
            
    df_stats = pd.DataFrame(stats)
    
    color_map = {
        'Working': '#3498DB',
        'Driving': '#F39C12',
        'Waiting': '#BDC3C7',
        'Setup': '#16A085',
        'Idle Day': '#ECECEC',
        'Idle (Shift End)': '#D6DBDF'
    }
    
    fig = px.bar(df_stats, x="Minutes", y="Technician", color="Activity", 
                 title="Technician Utilization (Shift Breakdown)", 
                 orientation='h', color_discrete_map=color_map)
                 
    fig.update_layout(
        template="plotly_white",
        barmode='stack',
        xaxis_title="Time spent (Minutes)",
        yaxis_title="Technician"
    )
    
    fig.add_vline(x=parameters['SHIFT_END_MIN'], line_width=3, line_dash="dash", line_color="black")
    fig.show()


# 3a. Visualisierung der Auslastung der Techniker

Wie bereits in der `evaluate`-Funktion implementiert, berechnen wir die Auslastung jedes Technikers als Summe aus Fahrzeit, Arbeitszeit, Wartezeit und Rüstzeit. Diese Informationen können wir nutzen, um die Auslastung der Techniker zu visualisieren. Die untenstehende Grafik zeigt die Auslastung der Techniker, aufgeteilt in Fahrzeit, Arbeitszeit, Wartezeit und Rüstzeit. Jeder Balken repräsentiert einen Techniker, und die Farben unterscheiden die verschiedenen Zeitkomponenten.

Folgende Erkenntnisse können aus der Grafik gezogen werden:

- Der Meister Techniker am Depot D1 (Techniker mit ID 6) hat mit 62 Minuten die geringste Auslastung. Dieser Techniker hat nur einen Auftrag zugewiesen bekommen, der relativ nahe am Depot liegt, was zu kurzen Fahrzeiten führt. Durch seinen hohen Stundensatz (72 EUR/h) könnte es sein, dass der Algorithmus versucht hat, ihn möglichst wenig einzusetzen, um Kosten zu sparen. Der Meister Techniker am Depot D0 (Techniker mit ID 7) mit hat 208 Minuten ebenfalls eine geringe Auslastung, ihm wurden 2 Aufträge zugewiesen. Wir vermuten, dass der Algorithmus beide Techniker aufgrund ihrer hohen Stundensätze möglichst wenig eingesetzt hat, um die Gesamtkosten zu minimieren. Unsere Vermutung wird außerdem dadurch gestützt, dass beide Techniker nur Aufträge mit den benötigten Qualifikationen `PLUMB` und `ELECTRIC` zugewiesen bekommen haben, die beide Techniker erfüllen können. Es wurden keine Aufträge mit der Qualifikation `DELIVERY` zugewiesen, da diese von allen Technikern ausgeführt werden können und somit anderen Technikern zugeordnet wurden, um die Kosten zu optimieren.
- Die Techniker mit der ID 0 und 3 wurden ebenfalls selten eingesetzt, da sie jeweils nur 2 Aufträge zugewiesen bekommen haben. Die andere Techniker haben im Gegensatz dazu 4 oder 5 Aufträge zugewiesen bekommen. Wir vermuten, dass die Aufträge so verteilt wurden, damit möglichst wenig Wartezeiten entstehen. Würde man die Aufträge gleichmäßig verteilen, könnte es sein, dass einige Techniker längere Wartezeiten haben, da sie nicht kontinuierlich Aufträge zugewiesen bekommen. Durch die ungleichmäßige Verteilung der Aufträge könnte der Algorithmus versucht haben, die Wartezeiten zu minimieren, indem er die Aufträge so zuweist, dass die Techniker möglichst kontinuierlich beschäftigt sind.

In [11]:
if 'best_individual' in locals():
    visualize_time_plan(best_individual)
else:
    print("Run the genetic algorithm first to get the best_individual.")
    
if 'best_individual' in locals():
    plot_technician_utilization(best_individual)

Anhand der unterstehenden Grafik können wir die Auslastung der Techniker visualisieren: Der größte Kostenfaktor ist wenig überraschend die Arbeitszeit. Interessant ist hingegen, dass der zweitgrößte Kostenfaktor die Wartezeit ist. Dies erklärt, warum der Algorithmus versucht hat, die Aufträge so zu verteilen, dass möglichst wenig Wartezeiten entstehen. Trivial zu erwähnen ist wohl, dass es keine Strafzahlungen gab, da diese eine optimale Lösung nicht konsituieren könnte. Allgemein lässt sich sagen, dass etwa die Hälfte der Einnahmen am Ende als Gewinn übrig bleibt, während die andere Hälfte für die Kosten (Arbeitszeit, Fahrzeit, Wartezeit und Rüstzeit) aufgewendet wird. 

In [12]:
if 'best_individual' in locals():
    plot_cost_breakdown(best_individual)

# 3b. Was passiert, wenn Sie die Verspätungsstrafen (penalty_eur_per_min) bei den VIP Kunden global verdoppeln?

Um diese Frage zu beantworten, haben wir die `evaluate`-Funktion angepasst, um die Strafzahlungen für verspätete VIP-Aufträge zu verdoppeln. Das bedeutet, dass die Kosten für Verspätungen bei VIP-Kunden nun deutlich höher sind, was den Algorithmus dazu zwingt, diese Aufträge mit höherer Priorität zu behandeln.

In [13]:
def evaluate_vip_penalty(individual):
    # Berechnung des Gesamtgewinns basierend auf der Vergütung und den Kosten der Techniker
    total_revenue = 0
    
    # Schnelles Gruppieren der Aufträge pro Techniker
    assigned_to_tech = {t_id: [] for t_id in tech_dict.keys()}
    for job_id, tech_id in individual:
        assigned_to_tech[tech_id].append(job_id)
        
    for tech_id, assigned_jobs in assigned_to_tech.items():
        revenue_per_technician = 0
        current_time = 0
        paid_working_time = 0
        
        technician = tech_dict[tech_id]
        current_location = technician.start_node
        
        if not assigned_jobs:
            continue

        # Ultraschnelles Sortieren der Jobs nach geplanter Startzeit mithilfe des Dictionaries
        assigned_jobs.sort(key=lambda j: job_dict[j]['planned_start_min'])
                
        for i, job_id in enumerate(assigned_jobs):
            job_info = job_dict[job_id]
            
            # Berechnung der Fahrzeit zum nächsten Auftrag
            job_location = job_info['node_id']
            travel_time = travel_times_dict.get((current_location, job_location), 0)
            current_time += travel_time
            paid_working_time += travel_time
            
            # Wartezeit vor Ort oder Abfahrt verzögern (beim ersten Job)
            planned_start_min = job_info['planned_start_min']
            
            if i == 0 and current_time < planned_start_min:
                # Techniker wartet zu Hause, fährt erst passend los
                current_time = planned_start_min
            else:
                if current_time < planned_start_min:
                    # Wartezeit vor Ort (gilt als bezahlte Arbeitszeit)
                    wait_time = planned_start_min - current_time
                    current_time = planned_start_min
                    paid_working_time += wait_time
                elif current_time > planned_start_min:
                    if job_info['vip'] == 1:
                        compensation = (current_time - planned_start_min) * job_info['penalty_eur_per_min'] * 2  # Doppelte Strafe für VIP-Aufträge
                    else:
                        compensation = (current_time - planned_start_min) * job_info['penalty_eur_per_min']
                    revenue_per_technician -= compensation  # Abzug für verspäteten Start

            # Berechnung der Arbeitszeit für den Auftrag
            current_time += job_info['duration_min']
            paid_working_time += job_info['duration_min']
            current_location = job_location
            
            # Skills prüfen
            required_skill = job_info['required_skill']
            if required_skill in technician.skills:
                revenue_per_technician += job_info['revenue_eur']
            else:
                revenue_per_technician -= penalty  # Strafe für unqualifizierte Techniker
            
            # Rüst-/Dokumentationszeit zwischen den Aufträgen
            if i < len(assigned_jobs) - 1:
                current_time += parameters['SETUP_BETWEEN_JOBS_MIN']
                paid_working_time += parameters['SETUP_BETWEEN_JOBS_MIN']
                
        # Berücksichtigung der Arbeitszeit des Technikers
        if current_time > parameters['SHIFT_END_MIN']:
            revenue_per_technician -= penalty  # Strafe für Überschreitung der Schicht
            
        # Abzug der Kosten basierend auf der tatsächlichen Arbeitszeit
        revenue_per_technician -= (technician.cost_eur_per_h / 60) * paid_working_time  
            
        total_revenue += revenue_per_technician    
        
    return total_revenue,

toolbox.register("evaluate", evaluate_vip_penalty)

best_individual_vip, logbook_vip = main()

toolbox.register("evaluate", evaluate)

gen	evals	avg    	min    	max    	std    	median 
0  	1000 	-172715	-431981	2641.58	83999.6	-206086
1  	806  	-129083 	-326834	2641.58	69908.9	-112110 
2  	813  	-109025 	-315477	2641.58	72335.5	-108513 
3  	810  	-92504.9	-313603	2641.58	73318.8	-105703 
4  	809  	-78885.4	-311902	2676.43	70147.7	-103858 
5  	763  	-66077.3	-304711	2676.43	69063.4	-100557 
6  	784  	-52515  	-224392	2676.43	64432.5	-5343.7 
7  	781  	-45052.2	-222293	2682.38	61682.6	-3242.71
8  	796  	-35387.7	-218171	2682.38	57619  	-1542.55
9  	799  	-28072.2	-214870	2682.38	52988.7	-369.417
10 	770  	-20837.9	-210528	2733.98	46048.7	864.617 
11 	779  	-12542.9	-204466	2733.98	37718.5	1797.78 
12 	786  	-8035.71	-203308	2733.98	32186.1	2321.43 
13 	795  	-6277.12	-200600	2733.98	28526.2	2530.41 
14 	789  	-4738.01	-109341	2733.98	25813.3	2641.58 
15 	803  	-8737.85	-202437	2733.98	32394.5	2641.58 
16 	799  	-8071.06	-200149	2733.98	31201.1	2641.58 
17 	768  	-7777.21	-204643	2733.98	32560.4	2651.18 
18 	783  	-8800.

Es ist zu beobachten, dass in beiden finalen, besten Lösungen (mit und ohne verdoppelte VIP-Strafe) **keine Strafzahlungen** anfallen (Strafen = 0), da der Algorithmus in beiden Fällen ausreichend Zeit findet, um Verspätungen komplett zu vermeiden. 

Dennoch führt die Verdopplung der VIP-Strafen zu einem **anderen optimalen Einsatzplan** mit leicht **geringerem Gesamtgewinn**. 
* **Ohne VIP-Fokus:** Gewinn ca. 3024,80 € (Geringere Reise- und Arbeitskosten)
* **Mit VIP-Fokus:** Gewinn ca. 2996,46 € (Leicht höhere Reise- und Arbeitskosten, dafür weniger Wartezeit)

**Warum passiert das, obwohl es am Ende gar keine Verspätungen gibt?**
Die verdoppelte Strafe bei VIP-Kunden verändert den *Evolutionspfad* drastisch. In den frühen Generationen und bei Mutationen werden Individuen, die bei VIP-Kunden auch nur minimal zu spät kommen, nun extrem hart bestraft und aussortiert. Dies erzeugt einen viel höheren Selektionsdruck, VIP-Aufträge "auf Nummer sicher" und mit großen zeitlichen Puffern einzuplanen (bzw. früher anzufahren). Der Algorithmus konvergiert dadurch in ein anderes, lokales Optimum. Dieses gewährleistet zwar ebenfalls Pünktlichkeit bei allen Aufträgen, ordnet die Rüstzeiten und Fahrten aber leicht ineffizienter an.

# 3c. Welche Parameter des EA sind besonders sensitiv?

1. **Mutationswahrscheinlichkeit auf Gen-Ebene (`mutpb = 0.3`)**  
   Dieser Parameter bestimmt, mit welcher Wahrscheinlichkeit die Zuordnung eines einzelnen Jobs innerhalb eines mutierten Individuums geändert wird. Er ist extrem sensitiv: Ist er zu hoch (z.B. > 0.3), wird die gesamte zeitliche (und sinnvolle) Struktur, die durch Selektion aufgebaut wurde, zerstört ("Random Walk"). Ist der Wert zu niedrig, verharrt der Algorithmus in dem lokalen Optimum, welches durch die initiale Greedy-Heuristik vorgegeben wurde.

2. **Kombination aus Turniergröße (`tournsize = 3`), Crossover (`cxpb = 0.7`) und Elitismus (`ELITISM_SIZE = 5`) (Selektionsdruck)**  
   Da wir in den ersten Generationen eine Mischung aus stark bestraften Zufallslösungen (Fitness stark im Minus) und 25 soliden heuristischen Lösungen haben, breiten sich die "guten" Gene durch den Selektionsdruck extrem schnell aus. Ein höherer `tournsize`-Wert würde zu vorzeitiger Konvergenz führen. Der Crossover-Wert von 70% ist notwendig, um die Eigenschaften der Heuristiken im gesamten Genpool zu mischen.

3. **Strafparameter (`penalty = 100000`)**  
   Die gewaltigen Strafen für Schichtüberschreitung (>600 Min) oder unzureichende Skills verändern die Fitnesslandschaft in ein Feld mit steilen Klippen. Jedes Individuum, das diese Restriktionen verletzt, "stirbt" sofort aus. In Konsequenz meidet der EA den Grenzbereich der maximalen Schichtdauer extrem schnell. Ein zu geringer Penalty würde dazu führen, dass der EA Überstunden in Kauf nimmt, um anderweitig Gewinne zu maximieren.

4. **Anteil des Heuristic-Seedings (Die ersten 25 von 1000 Individuen)**  
   Dies ist kein klassischer DEAP-Parameter, aber architektonisch hochsensitiv. Das Problem ("Finde Routen, die das strenge 600-Minuten-Limit nicht verletzen und nicht zu spät kommen") ist mit reiner Zufallsinitialisierung nur mit viel mehr Generationen bewältigbar, da die meisten rein zufälligen Abläufe in den massiven Strafen enden. Der Algorithmus ist für eine schnelle Konvergenz zum Optimum abhängig davon, dass diese ersten 25 Individuen als "gutes genetisches Material" den Basisvektor für die Population bilden.

# 3d. Alternativer Ansatz: Turnier- vs. Bestenselektion

Um die Auswirkungen der Selektionsstrategie zu beobachten, überschreiben wir die Elternselektion (Mating Pool) im Algorithmus. Bisher haben wir **Turnierselektion (`selTournament`)** verwendet. Nun testen wir **Bestenselektion (`selBest`)** für die Auswahl der Eltern.

In [14]:
# Änderung der Selektionsstrategie auf "Bestenselektion"
toolbox.unregister("select")

# Echte Bestenselektion (Truncation Selection): 
# Nimmt die Top 50 Individuen und füllt den 1000er Pool durch Duplikate auf.
def selBestReplicated(individuals, k, count_best=50):
    best = tools.selBest(individuals, count_best)
    return [random.choice(best) for _ in range(k)]

toolbox.register("select", selBestReplicated, count_best=50)

print("Starte EA mit echter Bestenselektion (erzeugt hohen Selektionsdruck)...")
best_individual_best_sel, logbook_best_sel = main()

# EA auf Turnierselektion zurücksetzen
toolbox.unregister("select")
toolbox.register("select", tools.selTournament, tournsize=3)

Starte EA mit echter Bestenselektion (erzeugt hohen Selektionsdruck)...
gen	evals	avg    	min    	max    	std    	median 
0  	1000 	-165371	-421811	2641.58	82314.8	-203034
1  	796  	-58331.5	-306276	2641.58	70084.7	-3672.98
2  	809  	-36545.5	-210777	2641.58	59398.6	802.992 
3  	801  	-25899.4	-310313	2657.78	53954.5	2495.47 
4  	778  	-6158.59	-201504	2700.35	29624  	2641.58 
5  	786  	-7106.91	-199418	2700.35	30369.4	2641.58 
6  	781  	-12257.3	-199425	2706.95	35982  	2677.58 
7  	787  	-9110.77	-201537	2708.1 	33251.1	2700.35 
8  	792  	-7291.53	-201842	2714.7 	32009.5	2700.35 
9  	780  	-7023.51	-199711	2714.7 	30056.7	2706.95 
10 	798  	-4894.96	-200511	2714.7 	28188.1	2714.7  
11 	783  	-7336.03	-201533	2714.7 	31417.5	2714.7  
12 	778  	-4382.78	-107264	2714.7 	25491.8	2714.7  
13 	811  	-6198.42	-200961	2714.7 	29491.1	2714.7  
14 	786  	-6057.74	-201448	2714.7 	28945.9	2714.7  
15 	778  	-4697.29	-110694	2714.7 	26015.6	2714.7  
16 	785  	-6652.84	-200673	2714.7 	30092  	2714.

In [15]:
plot_fitness_evolution(logbook_best_sel, "Interaktive Fitness-Entwicklung mit Bestenselektion (Erste 50 Generationen)")
plot_fitness_evolution(logbook, "Interaktive Fitness-Entwicklung mit Turnierselektion (Erste 50 Generationen)")

### Beobachtung: Turnierselektion vs. Bestenselektion

Die Turnierselektion führt zu einer stochastischen Auswahl, bei der auch Individuen mit schlechterer Fitness eine Chance haben, Eltern zu werden. Dies fördert die genetische Vielfalt und ermöglicht es dem Algorithmus, aus lokalen Optima auszubrechen. In der Praxis sehen wir, dass die Fitnesskurven mit Turnierselektion eine langsamere und stetigere Verbesserung zeigen als mit Bestenselektion. Die Bestenselektion hingegen wählt immer die besten Individuen aus, was zu einer schnelleren Konvergenz führt, aber auch das Risiko birgt, in einem lokalen Optimum stecken zu bleiben. In unserem Fall führt die Bestenselektion zu einer schnelleren Verbesserung der Fitness in den ersten Generationen, aber nach einigen Generationen bleibt die beste Fitness konstant, während die Turnierselektion weiterhin eine kleine Verbesserung ermöglicht. So konvergiert die Bestenselektion bei einer Fitness von 3015.88 €, während die Turnierselektion nach 1000 Generationen eine bessere Lösung mit 3024.77 € erreicht.

# 3e. Wo sehen Sie die kritischen Parameter für das Unternehmen? Haben Sie Vorschläge?

Aus rein betriebswirtschaftlicher Sicht decken die Modellierung und die grafischen Auswertungen (insbesondere die der Wartezeiten und Auslastung) einige kritische Hebel und Ineffizienzen für das Unternehmen auf:

**1. Starre, punktgenaue Startzeiten (`planned_start_min`)**
*   **Kritik:** Aktuell dürfen Aufträge nicht vor der fest geplanten Zeit begonnen werden. Wie in der Auslastungs-Grafik und beim Wasserfalldiagramm (Aufgabe 3a) klar zu sehen, führt das zu *massiven unproduktiven Wartezeiten*, die das Unternehmen voll bezahlen muss (da Techniker nach Zeit bezahlt werden).
*   **Vorschlag:** Einführung von **Service-Zeitfenstern (Time Windows)** bei den Kunden anstelle starrer Startzeiten (z. B. "am Dienstag zwischen 10 und 12 Uhr" statt exakt "10:30 Uhr"). Ist ein Techniker durch schnelles Fahren oder einen kurzen vorherigen Job früher da, kann er direkt starten. Dies minimiert bezahlten Leerlauf drastisch und erhöht die Zahl der machbaren Aufträge pro Tag.

**2. Die strikte Schichtgrenze (`SHIFT_END_MIN` = 600 Minuten)**
*   **Kritik:** Das System hat eine "Hard Constraint" bei 600 Minuten. Werden 601 Minuten gebraucht, gibt es hypothetische 100.000 € Strafe. Das führt dazu, dass gegen Schichtende oft noch (bezahlte) Zeit übrig ist, die aber z. B. für einen 40-Minuten-Auftrag nicht mehr ganz reicht. Der Tag bleibt suboptimal gefüllt.
*   **Vorschlag:** Einführung eines **Soft-Constraint-Modells für Überstunden**. Anstatt den Tag bei 600 Minuten rigide abzuschneiden, könnten Überstunden erlaubt werden, die jedoch teurer abgerechnet werden (z. B. 1,5- oder 2-facher Stundensatz). Wenn ein lukrativer Auftrag dadurch noch erledigt werden kann (oder teure VIP-Strafen für den nächsten Tag abgewendet werden), ist dies betriebswirtschaftlich oftmals der deutlich bessere Weg.

**3. Starre Rüstzeit zwischen Jobs (`SETUP_BETWEEN_JOBS_MIN` = 10 Minuten)**
*   **Kritik:** 10 Rüst-Minuten summieren sich bei z.B. 6 Aufträgen auf eine volle Stunde bezahlte unproduktive Zeit pro Techniker am Tag.
*   **Vorschlag:** Geschäftsprozessoptimierung. Das Unternehmen sollte versuchen, diese Zeiten durch Digitalisierung (z.B. mobile App, Speech-to-Text Dokumentation auf dem Weg zum Wagen) auf 3-5 Minuten zu halbieren. Jede gewonnene Arbeitsminute wandert im Full-Scale-Betrieb quasi direkt in den Unternehmensgewinn.

**4. Suboptimale Skalierung teurer Fachkräfte (Zusammenspiel `Skills` & `cost_eur_per_h`)**
*   **Kritik:** Die teuren Fachkräfte (z. B. Meister) hatten in der Analyse die kürzesten Einsatzzeiten. Da sie einen sehr hohen Stundensatz haben, versucht der Algorithmus sie möglichst nicht für Standard-Aufträge (z.B. `DELIVERY`) zu priorisieren, verbucht sie aber trotzdem für den Zeitraum.
*   **Vorschlag:** Erstens sollte das Preismodell (Umsatz pro Auftrag `revenue_eur`) für Tätigkeiten, die echte Meister erfordern (`PLUMB` / `ELECTRIC`), deutlich teurer abgerechnet werden, damit die Gewinnmarge die teure Arbeitskraft rechtfertigt. Zweitens könnten günstige Techniker gezielt weitergebildet werden ("Upskilling"), um die Last besser zu verteilen und auf teure Meisterstellen weitgehend verzichten zu können.